In [0]:
dbutils.widgets.text("catalog", "vp_rm20040829")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

##Stations

In [0]:
#create table with comment
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.stations
COMMENT 'This table contains the raw stations data'
""")

#copy into {catalog}.bronze.stations
spark.sql(f"""
copy into {catalog}.bronze.stations
from (SELECT *,
    _metadata.file_path as _source_file,
    current_timestamp() as _ingested_at
    FROM "/Volumes/{catalog}/landing/raw/stations/")
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
%sql
select * from vp_rm20040829.bronze.stations

##Tariffs

In [0]:
#create table {catalog}.bronze.tariffs
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.tariffs
COMMENT 'This table contains the raw tariffs data'
""")

#copy into {catalog}.bronze.tariffs
spark.sql(f"""
copy into {catalog}.bronze.tariffs
from (SELECT *,
    _metadata.file_path as _source_file,
    current_timestamp() as _ingested_at
    FROM "/Volumes/{catalog}/landing/raw/tariffs/")
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
%sql
select * from vp_rm20040829.bronze.tariffs

##Members

In [0]:
#create table {catalog}.bronze.tariffs
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.members
COMMENT 'This table contains the raw members data'
""")

#copy into {catalog}.bronze.members
spark.sql(f"""
copy into {catalog}.bronze.members
from (SELECT *,
    _metadata.file_path as _source_file,
    current_timestamp() as _ingested_at
    FROM "/Volumes/{catalog}/landing/raw/members/")
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
%sql
select * from vp_rm20040829.bronze.members

##Chargers

In [0]:
#create table {catalog}.bronze.chargers
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.chargers
COMMENT 'This table contains the raw chargers data'
""")

#copy into {catalog}.bronze.chargers
spark.sql(f"""
copy into {catalog}.bronze.chargers
from (SELECT *,
    _metadata.file_path as _source_file,
    current_timestamp() as _ingested_at
    FROM "/Volumes/{catalog}/landing/raw/chargers/")
FILEFORMAT = JSON
FORMAT_OPTIONS ("multiline" = "true")
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
%sql
select * from vp_rm20040829.bronze.chargers

##Sessions

In [0]:
#ingest into {catalog}.bronze.session using autoloader
(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/ops/pipeline_state")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .load(f"/Volumes/{catalog}/landing/raw/sessions/")
  .withColumn("_source_file", col("_metadata.file_path"))
  .withColumn("_ingested_at", current_timestamp())
  .writeStream
  .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/checkpoint")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable(f"{catalog}.bronze.sessions")
)

In [0]:
%sql
select * from vp_rm20040829.bronze.sessions